# Cobblemon Atlas｜烈空座物理攻擊鏡頭跳動診斷

本次只比較 48 幀固定鏡頭與安全鏡頭的動作中心位移，找出第 1→2 幀及中段跳動來源。保留原本 WebP，不覆蓋正式圖鑑。

In [ ]:
from pathlib import Path
import subprocess,sys,zipfile,shutil,json
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Cobblemon_Atlas')
WORK=Path('/content/cobblemon-bedrock')
SITE=Path('/content/cobblemon-atlas')
BRANCH='feat/animation-manifest-scanner-phase1'
assert ROOT.is_dir()

In [ ]:
if not (SITE/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','--branch',BRANCH,'https://github.com/hwh0410/cobblemon-atlas.git',str(SITE)],check=True)
subprocess.run(['git','-C',str(SITE),'sparse-checkout','set','tools','notebooks'],check=True)
subprocess.run(['git','-C',str(SITE),'pull','--ff-only','origin',BRANCH],check=True)
print('Git SHA:',subprocess.check_output(['git','-C',str(SITE),'rev-parse','--short','HEAD'],text=True).strip())

In [ ]:
WORK.mkdir(parents=True,exist_ok=True)
for name in ('merged_pack','development_resource_packs'):
    expected=WORK/name
    if not expected.is_dir():
        archive=ROOT/'source_packs'/(name+'.zip')
        assert archive.is_file(),archive
        with zipfile.ZipFile(archive) as z:
            for item in z.infolist():
                assert (WORK/item.filename).resolve().is_relative_to(WORK.resolve()),'Unsafe zip path'
            z.extractall(WORK)
    assert expected.is_dir(),expected
(WORK/'tools').mkdir(exist_ok=True)
shutil.copy2(ROOT/'source_packs/render_portraits.py',WORK/'tools/render_portraits.py')
print('模型素材已準備完成')

In [ ]:
resolved=ROOT/'reports/resolver-validation/resolved-forms.json'
out=ROOT/'reports/resolver-validation/physical-attack-jump-diagnosis.json'
subprocess.run([sys.executable,str(SITE/'tools/diagnose_rayquaza_camera_jumps.py'),'--repo',str(WORK),'--resolved',str(resolved),'--out',str(out)],check=True)
a=json.loads(out.read_text())
print('固定鏡頭最大跳動：',a['fixed_top_jumps'][:3])
print('安全鏡頭最大跳動：',a['safe_top_jumps'][:3])
print('疑似鏡頭放大跳動：',[x for x in a['jump_cases'] if x['likely_camera_amplification']])
print('Report:',out)